In [3]:
# Chunk 2: Data Cleaning & Exploratory Data Analysis (EDA) — Summary
'''
##1. Data Cleaning
- **Removed cancelled/fraud shipments**: Dropped 7,754 rows where `Delivery Status = 'Shipping canceled'`, 
  since these orders were never actually shipped (status: CANCELED or SUSPECTED_FRAUD), and their 
  `Late_delivery_risk = 0` label was misleading (falsely implying "on-time" delivery).
- **Removed PII columns**: Dropped `Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, 
  `Customer Street` — not useful for modeling and a privacy consideration.
- **Removed heavily missing columns**: Dropped `Product Description` (100% missing) and `Order Zipcode` 
  (86% missing).
- **Handled remaining missing values**: Dropped 3 rows with missing `Customer Zipcode`.
- **Fixed data types**: Converted `order date (DateOrders)` and `shipping date (DateOrders)` to proper 
  datetime format.
- **Final cleaned dataset**: 172,762 rows × 46 columns.

## 2. Target Variable Verification
- Verified `Late_delivery_risk` (0/1) against `shipping_gap = Days for shipping (real) - Days for 
  shipment (scheduled)`.
- Found a perfect separation: `shipping_gap ≤ 0` always corresponds to `Late_delivery_risk = 0`, and 
  `shipping_gap ≥ 1` always corresponds to `Late_delivery_risk = 1`.
- **Conclusion**: Target variable is reliable and consistent — safe to use directly for modeling.
- **Data leakage identified**: `Days for shipping (real)` and `shipping_gap` cannot be used as model 
  features, since they are only known after delivery occurs and directly determine the target.

## 3. Class Imbalance Check
- Late (1): 98,976 orders (57.3%)
- Not Late (0): 73,786 orders (42.7%)
- Dataset is reasonably balanced — no special resampling (SMOTE, class weighting) required for modeling.

## 4. Business Question Analysis (EDA Findings)
| Factor | Delay Risk Range | Impact Strength | Key Finding |
|---|---|---|---|
| Shipping Mode | 40% – 100% | 🔴 Very Strong | Tight scheduled delivery windows (e.g., First Class) drive artificially high "late" rates, even when actual delivery speed is comparable to Standard Class |
| Product Price | 40% – 70% | 🟡 Moderate | Non-linear relationship — mid-range priced products (₹806–1204) are riskiest; premium products (₹1600+) are safest |
| Product Category | 58% – 69% | 🟡 Moderate | Niche/specialty categories (Golf Bags & Carts, Lacrosse) show higher delay risk than high-volume categories (Books, Electronics) |
| Order Region | 51% – 59% | 🟢 Weak | Minimal variation across regions — geography is not a strong delay driver |
| Customer State | 59% – 63% | 🟢 Weak | Consistent with region-level finding; sample sizes (159–2,298 orders) verified as sufficiently reliable |
| Customer Segment | 57% – 58% | ⚪ Negligible | No meaningful difference between Consumer, Corporate, and Home Office segments |
| Order Profit Per Order | ₹21.6 vs ₹22.6 | ⚪ Negligible | Late delivery has no meaningful direct impact on recorded profit (profit is determined at order placement, not delivery outcome) |
| Discount Rate | 56% – 58% | ⚪ Negligible | No relationship between discount rate and delay risk |

## 5. Master Insight
Late delivery risk is primarily driven by **operational/logistics factors** (specifically, shipping 
mode scheduling), rather than customer demographics, geography, or pricing. The most actionable lever 
for reducing late delivery is setting more realistic scheduled delivery windows for fast-shipping modes 
(First Class, Second Class), rather than targeting specific regions, customer segments, or price tiers.

## 6. Known Limitations
- Order cancellation patterns (Business Question E) could not be explored, since cancelled/fraud orders 
  were removed during data cleaning to preserve target variable integrity.
- Financial impact analysis only captures direct/recorded profit; indirect costs of late delivery 
  (customer churn, negative reviews, complaint handling) are not present in this dataset.
  '''

'\n##1. Data Cleaning\n- **Removed cancelled/fraud shipments**: Dropped 7,754 rows where `Delivery Status = \'Shipping canceled\'`, \n  since these orders were never actually shipped (status: CANCELED or SUSPECTED_FRAUD), and their \n  `Late_delivery_risk = 0` label was misleading (falsely implying "on-time" delivery).\n- **Removed PII columns**: Dropped `Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`, \n  `Customer Street` — not useful for modeling and a privacy consideration.\n- **Removed heavily missing columns**: Dropped `Product Description` (100% missing) and `Order Zipcode` \n  (86% missing).\n- **Handled remaining missing values**: Dropped 3 rows with missing `Customer Zipcode`.\n- **Fixed data types**: Converted `order date (DateOrders)` and `shipping date (DateOrders)` to proper \n  datetime format.\n- **Final cleaned dataset**: 172,762 rows × 46 columns.\n\n## 2. Target Variable Verification\n- Verified `Late_delivery_risk` (0/1) against `shipping_g

In [4]:
'''Chunk 3: Feature Engineering — Summary

1. Lead time variance
Found Days for shipment (scheduled) is a fixed value per Shipping Mode (no variance) — not usable directly.
Calculated shipping_gap = actual − scheduled as the real signal.
Tested grouping by Department Name (flat), Category Name (flat, noisy), Order Region (real spread), Market (too coarse), Shipping Mode (strongest, cleanest signal).
Built features: mode_avg_delay, mode_delay_std, region_avg_delay, region_delay_std.

2. Seasonal/time-based features
Tested order_month, order_day_of_week, order_quarter, order_year.
All four showed flat, non-differentiating patterns (large samples, so not noise) — the "holiday season" and "weekend" hypotheses were tested and rejected.
Decision: no time-based features included in the model.

3. Supplier/reliability historical score
Built mode_reliability_score and region_reliability_score — expanding (before-only) averages per group, computed after sorting by date to avoid leakage.
Handled NaNs (first occurrence per group) by filling with overall mean.

4. Categorical encoding
One-hot encoded Shipping Mode (4 categories) → mode_First Class, mode_Same Day, mode_Second Class, mode_Standard Class.
Skipped one-hot encoding Order Region, Category Name, Department Name — either too high-cardinality or too weak a signal; their numeric engineered features already capture what matters.

Final feature set for modeling:
mode_avg_delay, mode_delay_std, mode_reliability_score, region_avg_delay, region_delay_std, region_reliability_score, mode_First Class, mode_Same Day, mode_Second Class, mode_Standard Class

Dataset: 172,762 rows × 60 columns.'''

'Chunk 3: Feature Engineering — Summary\n\n1. Lead time variance\nFound Days for shipment (scheduled) is a fixed value per Shipping Mode (no variance) — not usable directly.\nCalculated shipping_gap = actual − scheduled as the real signal.\nTested grouping by Department Name (flat), Category Name (flat, noisy), Order Region (real spread), Market (too coarse), Shipping Mode (strongest, cleanest signal).\nBuilt features: mode_avg_delay, mode_delay_std, region_avg_delay, region_delay_std.\n\n2. Seasonal/time-based features\nTested order_month, order_day_of_week, order_quarter, order_year.\nAll four showed flat, non-differentiating patterns (large samples, so not noise) — the "holiday season" and "weekend" hypotheses were tested and rejected.\nDecision: no time-based features included in the model.\n\n3. Supplier/reliability historical score\nBuilt mode_reliability_score and region_reliability_score — expanding (before-only) averages per group, computed after sorting by date to avoid leaka

In [5]:
'''Feature Engineering for Shortage Prediction — Summary

1. Checked for a real shortage/inventory signal — found none
Product Status was constant (0 for all 172,762 rows) — no variation, unusable.
Order Status had no shortage-specific category (only generic lifecycle statuses: COMPLETE, PENDING_PAYMENT, PROCESSING, etc.).
Conclusion: DataCo has no real stockout/shortage column — a proxy target was required.

2. Built the proxy target: shortage_risk
Aggregated Order Item Quantity by Product Name + week (order_week) → weekly_quantity.
Computed before-only, expanding mean and std of weekly demand per product (same leakage-safe technique as the delay model's reliability scores).
Flagged shortage_risk = 1 when a week's demand exceeded (rolling mean + 2×rolling std) for that product; NaNs (first 1-2 weeks per product, no baseline) filled with 0.
Result: ~5% positive rate (352 of 6,972 product-weeks) — merged back onto every order.

3. Caught a confound before trusting it
Category-level shortage rates showed a huge spread (0.00–0.59), but this tracked order volume, not category identity — low-volume categories looked artificially spikier due to noisier baselines (same low-sample-size trap from Chunk 3). Documented as a limitation, not used as a feature.

4. Final 4 engineered features:
product_demand_mean — product's typical weekly demand (before-only)
product_demand_std — how volatile that demand normally is
qty_deviation — how unusual this specific order's quantity is vs. the product's typical order size (z-score style; std=0 cases set to 0, not NaN)
product_order_history_count — expanding count of prior orders for that product (flags thin-history/noisy products)

5. Checkpoint saved: shortage_engineered_dataset.pkl/csv — 172,762 rows × 54 columns.'''

"Feature Engineering for Shortage Prediction — Summary\n\n1. Checked for a real shortage/inventory signal — found none\nProduct Status was constant (0 for all 172,762 rows) — no variation, unusable.\nOrder Status had no shortage-specific category (only generic lifecycle statuses: COMPLETE, PENDING_PAYMENT, PROCESSING, etc.).\nConclusion: DataCo has no real stockout/shortage column — a proxy target was required.\n\n2. Built the proxy target: shortage_risk\nAggregated Order Item Quantity by Product Name + week (order_week) → weekly_quantity.\nComputed before-only, expanding mean and std of weekly demand per product (same leakage-safe technique as the delay model's reliability scores).\nFlagged shortage_risk = 1 when a week's demand exceeded (rolling mean + 2×rolling std) for that product; NaNs (first 1-2 weeks per product, no baseline) filled with 0.\nResult: ~5% positive rate (352 of 6,972 product-weeks) — merged back onto every order.\n\n3. Caught a confound before trusting it\nCategor

In [6]:
'''Chunk 4: Model Training (Delay Prediction) — Summary

Train-test split — chronological (no shuffle) 80/20 split: 138,209 train / 34,553 test rows, preserving time order to avoid leakage from your rolling reliability features.
Baseline (Logistic Regression) — 0.70 accuracy, but recall on "Late" was only 0.54 (missed ~46% of real delays).
Random Forest (default) — severely overfit: 0.9999 train accuracy vs 0.637 test accuracy. Diagnosed by comparing train vs test accuracy directly.
Random Forest (tuned) — added max_depth=8, min_samples_leaf=50 to constrain tree growth. Result: train/test accuracy converged to 0.703/0.696 — overfitting resolved.
Key finding — once tuned, Random Forest performed almost identically to Logistic Regression (~0.70 accuracy, ~0.55 recall on "Late" both). Conclusion: the current 10-feature set is the bottleneck, not model choice — a real limitation worth documenting, not a failure.
Model saved — rf_model_tuned saved via joblib as delay_prediction_model.joblib (the chosen final model, for the overfitting-fix story and better robustness).'''

'Chunk 4: Model Training (Delay Prediction) — Summary\n\nTrain-test split — chronological (no shuffle) 80/20 split: 138,209 train / 34,553 test rows, preserving time order to avoid leakage from your rolling reliability features.\nBaseline (Logistic Regression) — 0.70 accuracy, but recall on "Late" was only 0.54 (missed ~46% of real delays).\nRandom Forest (default) — severely overfit: 0.9999 train accuracy vs 0.637 test accuracy. Diagnosed by comparing train vs test accuracy directly.\nRandom Forest (tuned) — added max_depth=8, min_samples_leaf=50 to constrain tree growth. Result: train/test accuracy converged to 0.703/0.696 — overfitting resolved.\nKey finding — once tuned, Random Forest performed almost identically to Logistic Regression (~0.70 accuracy, ~0.55 recall on "Late" both). Conclusion: the current 10-feature set is the bottleneck, not model choice — a real limitation worth documenting, not a failure.\nModel saved — rf_model_tuned saved via joblib as delay_prediction_model.j

In [7]:
'''Chunk 5 summary: (Model Training — Shortage Prediction)

No real shortage/inventory signal in DataCo — Product Status was constant, Order Status had no shortage-specific category. Built a proxy: shortage_risk flags a product-week where demand exceeded its historical (before-only) mean + 2×std — ~5% positive rate.
Confounding caught early — category-level shortage rates looked highly variable, but this was driven by order-volume noise (low-volume products spike more easily), not real category effects — same low-sample-size trap as Chunk 3.
4 features engineered: product_demand_mean, product_demand_std, qty_deviation, product_order_history_count.
Logistic Regression (class_weight='balanced'): 0.77 accuracy, 0.58 recall / 0.11 precision / 0.18 F1 on shortage class.
Threshold tuning (0.3–0.7): no meaningful improvement — flat precision across all thresholds, meaning the features lack strong class separation.
Random Forest (tuned, class_weight='balanced'): higher accuracy (0.82) but worse on the metric that matters — recall dropped to 0.18, F1 to 0.08. Rejected despite higher accuracy.
Final model saved: Logistic Regression — best F1 on the minority class, the honest choice over the higher-accuracy-but-less-useful Random Forest.'''

"Chunk 5 summary: (Model Training — Shortage Prediction)\n\nNo real shortage/inventory signal in DataCo — Product Status was constant, Order Status had no shortage-specific category. Built a proxy: shortage_risk flags a product-week where demand exceeded its historical (before-only) mean + 2×std — ~5% positive rate.\nConfounding caught early — category-level shortage rates looked highly variable, but this was driven by order-volume noise (low-volume products spike more easily), not real category effects — same low-sample-size trap as Chunk 3.\n4 features engineered: product_demand_mean, product_demand_std, qty_deviation, product_order_history_count.\nLogistic Regression (class_weight='balanced'): 0.77 accuracy, 0.58 recall / 0.11 precision / 0.18 F1 on shortage class.\nThreshold tuning (0.3–0.7): no meaningful improvement — flat precision across all thresholds, meaning the features lack strong class separation.\nRandom Forest (tuned, class_weight='balanced'): higher accuracy (0.82) but

In [ ]:
'''# Chunk 6: Database Setup (SQLite)
## What I built
A SQLite database (`data/supply_chain.db`) that stores the cleaned orders and the predictions from both of my models. SQL queries on top of it feed the dashboard and the API.

## Why a database?
- The dashboard needs to filter, sort and group predictions quickly.
- SQL lets me join orders with predictions without loading everything into pandas each time.
- It keeps the model code and the app code separate: the models write predictions once, and the app only reads them.

## How it works
1. `db_connection.py` opens the database. The path comes from `config.yaml`, so nothing is hardcoded.
2. The cleaned dataset is loaded into the `orders` table (172,762 rows, 46 columns).
3. I load the saved delay and shortage models (`.joblib`) and predict on the engineered features.
4. The predictions are saved to two tables: `delay_predictions` and `shortage_predictions`.
5. `queries.py` has reusable functions that the API (Chunk 8) and the dashboard (Chunk 9) will call.

## Tables
| Table | What it holds |
|---|---|
| `orders` | Cleaned order data (shipping mode, region, category, etc.) |
| `delay_predictions` | order_item_id, delay_prob, delay_pred, risk_level |
| `shortage_predictions` | order_item_id, shortage_prob, shortage_pred, risk_level |

All tables are joined on `order_item_id`.

## Risk levels
- **Delay:** based on probability (Low below 0.45, Medium 0.45 to 0.75, High above 0.75). The delay probabilities match the real late rate for each shipping mode, so fixed cut-offs work.
- **Shortage:** based on rank (top 10% = High, next 20% = Medium, rest = Low). The real shortage rate is only 2.9%, and I used `class_weight='balanced'`. That inflates the scores, so I treat them as a ranking score, not a true probability.

## SQL concepts used
SELECT, WHERE, JOIN (up to 3 tables), GROUP BY, ORDER BY, LIMIT, aggregate functions (COUNT, AVG, ROUND), and parameterised queries (`?`) to stay safe from SQL injection.

## Key results
- Delay by shipping mode (average probability vs real late %): First Class 1.00 vs 100%, Second Class 0.80 vs 79.8%, Same Day 0.47 vs 47.9%, Standard Class 0.40 vs 39.8%.
- 5,995 orders are High risk for both delay and shortage.
- Risk split: delay has 59,950 High (First and Second Class), shortage has 17,277 High (top 10%).

## Limitations (say these honestly)
- The delay model mostly learns the shipping mode, so region barely changes the prediction (0.53 to 0.59 across regions).
- Shortage uses a proxy target (demand spikes), and its scores are rankings, not probabilities.
- Predictions are made on historical data, so this is a decision-support demo, not live forecasting.

## Likely interview questions
- **Why SQLite and not PostgreSQL?** It needs no server, it is perfect for a single-user portfolio project, and the same SQL works on PostgreSQL later.
- **Why store predictions instead of predicting live?** The data is static, so predicting once and storing is faster and cheaper. With live data I would call the model through the API.
- **Why not use the 0.5 threshold?** For delay, the probabilities are calibrated, so I used probability bands. For shortage, balanced class weights inflate scores, so I used percentile ranking.
- **What does the JOIN do here?** It brings shipping mode, region and category from `orders` into the predictions tables.'''

In [ ]:
"""# Chunk 8: FastAPI Backend

## 1. What I built
A REST API that sits between the database / LLM and the dashboard. The dashboard never touches the database directly; it only calls the API.

Flow: Streamlit dashboard -> FastAPI (routes) -> queries.py -> SQLite
                                              -> briefing.py -> Groq LLM (with cache)

## 2. Why an API layer?
- Separation of concerns: the dashboard only shows data, the API owns the logic.
- Any other client (a mobile app, another service) can reuse the same API.
- It can be deployed on its own, and tested on its own.
- FastAPI gives free interactive docs at `/docs` (Swagger), good for demos.

## 3. Endpoints
| Endpoint | Method | What it does |
|---|---|---|
| `/predict/delay` | POST | Stored delay probability + risk level + facts for one order |
| `/predict/shortage` | POST | Stored shortage score + risk level + facts for one order |
| `/briefing` | POST | LLM summary: why this order is risky, what to do |
| `/whatif/questions` | GET | Preset what-if questions (from config.yaml) |
| `/whatif` | POST | LLM answer to a what-if question about one order |
| `/summary` | GET | KPI numbers (total orders, high delay, high shortage, high on both) |
| `/filters` | GET | Values for the dropdowns (shipping modes, regions) |
| `/orders/risky` | GET | The risk table, with filters + pagination (`limit`, `offset`) |
| `/charts/...` | GET | Data for each dashboard chart (added in Chunk 9) |

## 4. Key design decisions
- **Pydantic schemas (schemas.py)**
  - `Literal["delay", "shortage"]` for risk_type: wrong values are rejected with 422 automatically.
  - `Field(gt=0)` on order_item_id, `min_length` on the question.
  - `response_model` makes sure the API always returns the shape we promised.
- **APIRouter (routes.py)**: routes are kept apart from `main.py`, which only creates the app (title and version come from config.yaml).
- **Error handling**
  - 404 if the order does not exist.
  - 502 if the LLM service fails (with a friendly message; the real error goes to the log).
  - 422 for invalid input (automatic).
- **`_clean()` helper** converts numpy numbers from pandas into normal Python numbers so they can be turned into JSON.
- **Honest naming**: the database column is `shortage_prob`, but the API returns it as `shortage_score` because it is a ranking score, not a probability.
- **SQL safety**: all values go in as `?` parameters. Column names for ORDER BY come from a fixed dictionary, never from user input. This prevents SQL injection.
- **Predictions are precomputed**: all predictions were generated once and stored in SQLite. The API reads them; it does not run the model live.
- **LLM cache**: the `llm_briefings` table is keyed by (kind, entity_key, prompt_version, question). Same question on the same order = instant answer, no new LLM call, no extra cost.
- **Prompt versions in config.yaml**: changing the active version re-generates answers without a code change.

## 5. Testing
- pytest + FastAPI `TestClient`, 8 tests (valid calls, 404, 422, response shape).
- A test found a real bug: `order_item_id = 0` was returning 404 instead of 422. Fix: `gt=0` in the schema.
- Run with: `python -m pytest tests/ -v`

## 6. Likely interview questions
**Q: Why FastAPI and not Flask?**
It has built-in validation (Pydantic), auto docs, type hints and is fast. Less boilerplate for a data API.

**Q: What does Pydantic do here?**
It checks the incoming request and the outgoing response against a schema. Bad input is rejected before it reaches my code.

**Q: How do you stop SQL injection?**
Parameterised queries (`?`). Dynamic parts like ORDER BY come from a fixed dictionary.

**Q: Why cache the LLM responses?**
LLM calls are slow (about 3 seconds) and cost money. The same order and question always gives the same answer, so I store it once.

**Q: Why not predict live in the API?**
Simplicity and speed for a portfolio project. The honest trade-off: if new data arrives, predictions must be regenerated. Live prediction would be the next step.

**Q: How did you test the API?**
pytest with TestClient: success paths, 404, and validation errors (422). The tests even caught the `gt=0` bug.

## 7. Limitations (be honest)
- Predictions are batch-generated, not live.
- Only 8 tests; no load testing and no authentication.
- LLM answers depend on the prompt and the facts I pass in."""

In [ ]:
"""# Chunk 9: Streamlit Dashboard

## 1. What I built
An interactive dashboard with a glassmorphism look, plus a separate full-page What-if chat assistant. It talks to the FastAPI backend only (using `requests`). It never reads SQLite directly.

## 2. File structure
- `dashboard/app.py`: main page (sidebar, KPIs, charts, risk table)
- `dashboard/components/risk_table.py`: the paginated, selectable risk table
- `dashboard/components/charts.py`: all Plotly chart functions
- `dashboard/components/chat_interface.py`: helpers for calling the API from the chat page
- `dashboard/pages/whatif_chat.py`: the chatbot page (Streamlit multipage)
- `dashboard/assets/style.css`: glassmorphism styling

## 3. Main page features
- **Sidebar filters**: risk type (Both / Delay / Shortage), risk level, shipping mode, region.
- **4 KPI cards**: total orders, high delay risk, high shortage risk, high on both.
- **Filter chips** showing what is currently selected.
- **6 Plotly charts** (see section 4).
- **Risk table**: server-side pagination (limit/offset), rows per page, row selection.

## 4. Charts and which filters they use
| Chart | Shows | Filters used |
|---|---|---|
| Delay risk by shipping mode (bar) | Average delay probability per mode | Region |
| Top categories by shortage risk (horizontal bar) | % of orders at High shortage risk | Shipping mode, region |
| Delay risk levels (donut) | Low / Medium / High counts | Shipping mode, region |
| Shortage risk levels (donut) | Low / Medium / High counts | Shipping mode, region |
| Monthly orders + late delivery % (bars + line) | Real late rate over time, from the orders table | Shipping mode, region |
| Shipping mode x region (heatmap) | Average delay probability for every pair | None; the selection is outlined |

Why some filters do not apply:
- Risk level only filters the table. A donut that shows the Low/Medium/High split cannot also be filtered by level.
- The heatmap axes ARE shipping mode and region, so filtering would leave one cell. Instead I outline the selection.
- Risk type decides which charts are shown (Delay shows delay charts, Shortage shows shortage charts).

Chart details worth remembering:
- Shortage chart uses "% at High risk" because average scores were almost equal (0.73 to 0.74) and bars looked identical.
- `HAVING COUNT(*) >= 10` removes categories with too few orders; with filters, 1 or 2 orders produced fake 100% bars.
- Monthly chart: missing months are filled (reindex) so the line breaks instead of joining across gaps.

## 5. Streamlit concepts used
- `st.set_page_config(layout="wide")`, custom CSS through `st.markdown(unsafe_allow_html=True)`.
- `@st.cache_data(ttl=60)` on the API `get()` function so every rerun does not call the API again.
- Streamlit reruns the whole script on every interaction, so state that must survive goes in `st.session_state`.
- `st.dataframe(on_select="rerun", selection_mode="single-row")` for row click; the clicked row is saved as `session_state["selected_order"]`.
- Multipage app: a file in `pages/` becomes a page; `st.switch_page()` navigates; `session_state` is shared between pages.
- `st.chat_input` + `st.chat_message` for the chatbot UI.

## 6. What-if assistant page
- Opens from the sidebar button for the order selected in the table.
- **Briefing**: no question needed; one click gives a short AI summary (calls `/briefing`).
- **What-if**: pick a preset question or type any question (calls `/whatif`).
- Chat history is kept per (order, risk type) in session_state.
- `fmt()` puts every sentence on its own line (the regex ignores decimals like 0.60).
- First call takes about 3 seconds; repeated questions come from the cache.

Briefing vs What-if: briefing = automatic summary of why the order is risky. What-if = answer to a specific question. In both, the LLM only explains from the numbers it is given; it does not re-run the model.

## 7. Bugs I hit and fixed
- 422 error: the sidebar sent `Both` but the API accepts `both`. Fix: `.lower()`.
- Charts did not react to filters: the chart endpoints had no filter parameters. Fix: added optional `shipping_mode` / `region` and a shared `_where()` helper.
- Stale code after editing charts.py: Streamlit cached the old module. Fix: stop Streamlit fully, delete `__pycache__`, restart.
- LLM added "the model must be re-run for an exact value" to every answer. Cause: my own prompt (v4) said "always end with this". Fix: prompt v5 says to add it only when the question asks for a new or exact value. Lesson: a rule written as "always do X" gets over-applied; state the condition.

## 8. Insights from the dashboard (good to say in interviews)
- Delay risk is driven mainly by shipping mode: First Class is about 100% late, Second Class about 80%, Same Day about 48%, Standard Class about 40% (overall 57.3%).
- The heatmap shows rows differ a lot but columns barely differ, so region adds very little.
- Shortage score is a ranking score, not a probability. The real shortage rate is about 2.9%, so "High" means the top 10% by score.

## 9. Likely interview questions
**Q: Why does the dashboard call an API instead of the database?**
Separation of concerns. The UI stays simple, the logic lives in one place, and the backend can be reused or deployed separately.

**Q: How does Streamlit keep the selected row?**
`st.session_state`. Streamlit reruns the script on each interaction, so anything that must persist is stored there.

**Q: How did you handle thousands of rows in the table?**
Server-side pagination (limit and offset) plus a count query. The browser only gets 10 to 50 rows at a time.

**Q: What does "High risk" mean for shortage?**
It is a relative ranking (top 10% of orders by score), not a probability. I explain this in the table tooltip.

**Q: Did the LLM ever give a wrong or misleading answer?**
Yes. It repeated a "re-run the model" sentence in every answer. I traced it to my prompt, changed it to a conditional rule (v5), and documented the change.

**Q: What would you improve?**
Live predictions instead of precomputed ones, user login, and monitoring for model drift.

## 10. Limitations (be honest)
- Predictions are precomputed, so the dashboard is a view of stored results.
- Delay model mostly learns shipping mode; region barely matters.
- Shortage target is a proxy and its score is only for ranking.
- What-if answers are explanations from historical group averages, not new predictions.
- Chat history is lost when the browser is refreshed."""